<a href="https://colab.research.google.com/github/sairaj-pixel/Youtube-Clone/blob/main/final_year_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install streamlit pyngrok scikit-learn seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 104.8 MB/s eta 0:00:00


In [1]:
!pip install plotly

In [2]:
!ngrok config add-authtoken 3ATchstIeH4MOjb4bsD8COdkZBT_7CCJ4gyqq1EcB2Y1sR4D6

/bin/bash: line 1: ngrok: command not found


In [4]:
%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

st.set_page_config(layout="wide")

st.title("🌾 Crop Yield Prediction App")

# -----------------------------
# Load dataset
# -----------------------------

df = pd.read_csv("crop_yield_training_dataset.csv")

# Encode categorical variables
label_encoder = LabelEncoder()

df['Soil_Fertility'] = label_encoder.fit_transform(df['Soil_Fertility'])
df['Crop_Type'] = label_encoder.fit_transform(df['Crop_Type'])

# Features and target
X = df.drop(columns=['Yield'])
y = df['Yield']

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
model = LinearRegression()
model.fit(X_train, y_train)

# -----------------------------
# Manual Prediction Section
# -----------------------------

st.header("Enter Crop Conditions")

col1, col2 = st.columns(2)

with col1:
    temperature = st.number_input("Temperature (°C)", 0, 60, 25)
    rainfall = st.number_input("Rainfall (mm)", 0, 500, 120)

with col2:
    soil_moisture = st.number_input("Soil Moisture (%)", 0, 100, 40)

soil_fertility = st.selectbox("Soil Fertility", ["Good","Medium","Poor"])
crop_type = st.selectbox("Crop Type", ["Wheat","Rice","Corn"])

# Encode inputs manually
soil_map = {"Good":0,"Medium":1,"Poor":2}
crop_map = {"Wheat":0,"Rice":1,"Corn":2}

# Prediction
if st.button("Predict Yield"):

    input_data = np.array([[
        temperature,
        rainfall,
        soil_moisture,
        soil_map[soil_fertility],
        crop_map[crop_type]
    ]])

    prediction = model.predict(input_data)

    st.success(f"Predicted Yield: {prediction[0]:.2f} tons/hectare")

    st.session_state["show_dashboard"] = True


# -----------------------------
# Dashboard Button
# -----------------------------

if "show_dashboard" in st.session_state:

    if st.button("View Results Dashboard"):

        st.header("Model Evaluation Dashboard")

        # Predictions
        y_pred = model.predict(X_test)

        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        col1, col2 = st.columns(2)

        col1.metric("Mean Absolute Error", round(mae,3))
        col2.metric("R² Score", round(r2,3))

        st.divider()

        # True vs predicted
        st.subheader("True vs Predicted Yield")

        fig1, ax1 = plt.subplots()

        ax1.scatter(y_test, y_pred)

        ax1.plot(
            [y.min(), y.max()],
            [y.min(), y.max()],
            'k--'
        )

        ax1.set_xlabel("True Yield")
        ax1.set_ylabel("Predicted Yield")

        st.pyplot(fig1)

        # Feature importance
        st.subheader("Feature Importance")

        coefficients = pd.DataFrame(
            model.coef_,
            X.columns,
            columns=["Coefficient"]
        )

        fig2, ax2 = plt.subplots()

        sns.barplot(
            x=coefficients.index,
            y=coefficients["Coefficient"],
            ax=ax2
        )

        st.pyplot(fig2)

        # Yield distribution
        st.subheader("Yield Distribution")

        fig3, ax3 = plt.subplots()

        sns.histplot(df["Yield"], kde=True, ax=ax3)

        st.pyplot(fig3)

Overwriting app.py


In [5]:
from pyngrok import ngrok

!pkill streamlit
!pkill ngrok

!streamlit run app.py &>/dev/null &

public_url = ngrok.connect(8501)

print("Open this URL:", public_url)

ModuleNotFoundError: No module named 'pyngrok'